In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
print(list(uploaded.keys()))

Saving Q5-snap-dashboard-data.xlsx to Q5-snap-dashboard-data.xlsx
Saving Q5-snap-dashboard-statewide-totals.xlsx to Q5-snap-dashboard-statewide-totals.xlsx
['Q5-snap-dashboard-data.xlsx', 'Q5-snap-dashboard-statewide-totals.xlsx']


In [3]:
DATA_END = pd.Timestamp("2026-07-01")

# Identify each upload by name: "statewide" in the name = statewide file, otherwise = county file
targets = {"Q5-snap-dashboard-data": None, "Q5-snap-dashboard-statewide-totals": None}
for fname in uploaded:
    if not fname.lower().endswith(".xlsx"):
        continue
    key = "Q5-snap-dashboard-statewide-totals" if "statewide" in fname.lower() else "Q5-snap-dashboard-data"
    targets[key] = fname
print(targets)
assert all(targets.values()), f"Missing upload: {targets}"

out_files = []
for base, fname in targets.items():
    df = pd.read_excel(fname, sheet_name="Sheet0", parse_dates=["Report_Month"])
    trimmed = df[df["Report_Month"] <= DATA_END].copy()
    out_name = f"{base}_thru_Jul2026.xlsx"
    trimmed.to_excel(out_name, index=False)
    out_files.append(out_name)
    print(f"{base}: {len(df):,} → {len(trimmed):,} rows | "
          f"{trimmed['Report_Month'].min():%Y-%m} → {trimmed['Report_Month'].max():%Y-%m}")

{'Q5-snap-dashboard-data': 'Q5-snap-dashboard-data.xlsx', 'Q5-snap-dashboard-statewide-totals': 'Q5-snap-dashboard-statewide-totals.xlsx'}
Q5-snap-dashboard-data: 11,208 → 11,121 rows | 2016-01 → 2026-07
Q5-snap-dashboard-statewide-totals: 128 → 127 rows | 2016-01 → 2026-07


In [4]:
for out_name in out_files:
    chk = pd.read_excel(out_name, parse_dates=["Report_Month"])
    assert chk["Report_Month"].max() <= DATA_END, f"{out_name} still has months after Jul 2026"
    print(f"OK  {out_name}: {len(chk):,} rows, last month {chk['Report_Month'].max():%Y-%m}")

OK  Q5-snap-dashboard-data_thru_Jul2026.xlsx: 11,121 rows, last month 2026-07
OK  Q5-snap-dashboard-statewide-totals_thru_Jul2026.xlsx: 127 rows, last month 2026-07


In [5]:
for out_name in out_files:
    files.download(out_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>